# Google Calendar MCP Client Demo

This notebook demonstrates how to authenticate via the full OAuth dance and use the refactored `CalendarClient` directly. It coordinates with `MeetClient` and `DriveClient` internally.

In [ ]:
import os
import sys
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

sys.path.append('..')
from mcp_servers.google_calendar.app.calendar.calendar_client import CalendarClient
from mcp_servers.google_calendar.app.meet.meet_client import MeetClient


SCOPES = [
    'https://www.googleapis.com/auth/meetings.space.readonly',
    'https://www.googleapis.com/auth/calendar.events.readonly',
]

def get_credentials():
    creds = None
    if os.path.exists('token.json'):
        try:
            creds = Credentials.from_authorized_user_file('token.json', SCOPES)
        except ValueError:
            creds = None
            os.remove('token.json')
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('client_secret.json', SCOPES)
            creds = flow.run_local_server(port=51063)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

creds = get_credentials()
calendar_client = CalendarClient(creds)
meet_client = MeetClient(creds)

## Executing list_events

Fetch events using `calendar_client.list_events()`.

In [ ]:
events = calendar_client.list_events(max_events=5, date_min="2026-03-31", date_max="2026-04-01", query="MCP")

In [ ]:
events[0].conference_info

In [ ]:
event = events[0]

print(f"{event.start_time}")
print(f"{event.end_time}")

In [ ]:
event.location

In [ ]:
events[0].attendees[0].response_status.value

In [ ]:
event.attendees

In [ ]:
for attendee in event.attendees:
    print(f"{attendee.display_name = }, {attendee.email = }, {attendee.user_id = }")

In [ ]:
conference_sessions = meet_client.list_conference_sessions(conference_id="grd-gkxe-urd")

In [ ]:
conference_session_id = conference_sessions[1].conference_session_id

recording_id = conference_sessions[1].recordings.recording_ids[0]

recording_id

In [ ]:
conference_sessions[0].duration

In [ ]:
conference_sessions[1].duration

In [ ]:
participants = meet_client.get_participants_info(conference_session_id)

In [ ]:
for participant in participants:
    print(f"{participant.display_name = }, {participant.email = }, {participant.user_id = }")

In [ ]:
participants[0].time_in_meeting

In [ ]:
meet_client.get_recording_info(recording_id)

In [ ]:
conference_sessions[1].transcripts